### BLEU score

In [127]:
from collections import Counter

def get_ngrams(tokens, n):
    ngrams = []

    for i in range(len(tokens) - n + 1):
        ngram = tuple(tokens[i: i + n])
        ngrams.append(ngram)

    return Counter(ngrams)

In [128]:
def clipped_precision(reference, hypothesis, n):
    ref_ngrams = get_ngrams(reference, n)
    hypo_ngrams = get_ngrams(hypothesis, n)

    clipped_count = 0
    total_count = sum(hypo_ngrams.values())

    for ngram in hypo_ngrams:
        clipped_count += min(hypo_ngrams[ngram], ref_ngrams.get(ngram, 0))

    if total_count == 0:
        return 0

    return clipped_count / total_count

In [129]:
import math

def brevity_penalty(reference, hypothesis):
    ref_len = len(reference)
    hyp_len = len(hypothesis)

    if hyp_len == 0:
        return 0

    if hyp_len > ref_len:
        return 1

    else:
        return math.exp(1 - ref_len / hyp_len)

In [130]:
def bleu_score(reference, hypothesis, max_n=4):
    weights = [1/max_n] * max_n

    precisions = []

    for n in range(1, max_n + 1):
        p = clipped_precision(reference, hypothesis, n)
        precisions.append(p)

    if min(precisions) == 0:
        return 0

    log_precisions = sum(w * math.log(p) for w, p in zip(weights, precisions))

    bp = brevity_penalty(reference, hypothesis)

    return bp * math.exp(log_precisions)

In [131]:
def evaluate_bleu(data, decode_fn):
    scores_1, scores_2, scores_4 = [], [], []

    for src, ref in data:
        hyp_tokens = decode_fn(src)
        ref_tokens = ref.split()

        scores_1.append(bleu_score(ref_tokens, hyp_tokens, max_n=1))
        scores_2.append(bleu_score(ref_tokens, hyp_tokens, max_n=2))
        scores_4.append(bleu_score(ref_tokens, hyp_tokens, max_n=4))

    return (
        sum(scores_1) / len(scores_1),
        sum(scores_2) / len(scores_2),
        sum(scores_4) / len(scores_4),
    )

### Beam Search

In [132]:
import torch
import torch.nn.functional as F

def beam_search(model, sentence, eng_vocab, rus_vocab, device, beam_width=5, max_len=50, with_attention=False):
    model.eval()

    tokens = sentence.lower().split()
    src_idx = [eng_vocab.get(t, eng_vocab["<oov>"]) for t in tokens]
    src_tensor = torch.LongTensor(src_idx).unsqueeze(1).to(device)

    with torch.no_grad():
        if with_attention:
            encoder_outputs, hidden, cell = model.encoder(src_tensor)
        else:
            hidden, cell = model.encoder(src_tensor)
            encoder_outputs = None

    beams = [([rus_vocab["<sos>"]], 0.0, hidden, cell)]
    completed = []

    for _ in range(max_len):
        new_beams = []

        for seq, score, h, c in beams:
            last_token = seq[-1]

            if last_token == rus_vocab["<eos>"]:
                completed.append((seq, score))
                continue

            input_tensor = torch.LongTensor([last_token]).to(device)

            with torch.no_grad():
                if with_attention:
                    output, h_new, c_new = model.decoder(input_tensor, h, c, encoder_outputs)
                else:
                    output, h_new, c_new = model.decoder(input_tensor, h, c)

            log_probs = F.log_softmax(output, dim=1)
            # inside beam_search, right before topk:
            if len(seq) < 3:
                log_probs[0][rus_vocab["<eos>"]] = -1e9

            topk_probs, topk_idx = log_probs.topk(beam_width)

            for i in range(beam_width):
                token = topk_idx[0][i].item()
                new_score = score + topk_probs[0][i].item()
                new_seq = seq + [token]
                new_beams.append((new_seq, new_score, h_new, c_new))

        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_width]

    completed += [(seq, score) for seq, score, _, _ in beams]

    best_seq = sorted(completed, key=lambda x: x[1], reverse=True)[0][0]

    inv_vocab = {v: k for k, v in rus_vocab.items()}

    result = []
    for i in best_seq:
        word = inv_vocab[i]
        if word == "<eos>":
            break
        if word not in ["<sos>", "<pad>"]:
            result.append(word)

    return result

### Load test data

In [133]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\u0430-\u044f\u0451\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

test_data = []

with open("test_data.txt", encoding="utf-8") as f:
    for line in f:
        src, trg = line.strip().split("\t")
        # clean both sides the same way the models were trained on
        test_data.append((clean_text(src), clean_text(trg)))

print(test_data[:5])
print(f"Test set size: {len(test_data)}")

[('the old castle stands on the hill', 'старый замок стоит на холме'), ('tom was afraid of mary', 'том боялся мэри'), ('it was a blow to us', 'это было для нас ударом'), ('if possible id like to go home now', 'если можно я бы хотел теперь пойти домой'), ('i wasnt worried about tom', 'я не волновался за тома')]
Test set size: 100


### Vocabulary

In [134]:
def build_vocab(sentences):
    counter = Counter()

    for sent in sentences:
        words = sent.split()
        counter.update(words)

    vocab = {"<pad>": 0, "<oov>": 1}

    for word in counter:
        vocab[word] = len(vocab)

    return vocab

file_path = "rus.txt"
pairs = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) >= 2:
            pairs.append((parts[0], parts[1]))

eng_sentences = [clean_text(eng) for eng, rus in pairs]
rus_sentences = ["<sos> " + clean_text(rus) + " <eos>" for eng, rus in pairs]

eng_vocab = build_vocab(eng_sentences)
rus_vocab = build_vocab(rus_sentences)

print("English vocabulary size:", len(eng_vocab))
print("Russian vocabulary size:", len(rus_vocab))

English vocabulary size: 16319
Russian vocabulary size: 53282


### Seq2Seq Model (without attention)

In [135]:
import torch.nn as nn

class Encoder(nn.Module):

    def __init__(self, input_dim, emb_dim, hidden_dim, num_layers=1):
        super().__init__()

        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, num_layers)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.lstm(embedded)

        return hidden, cell


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, num_layers=1):
        super().__init__()

        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, num_layers)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, input, hidden, cell):
        input = input.unsqueeze(0)
        embedded = self.embedding(input)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc(output.squeeze(0))
        return prediction, hidden, cell


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        hidden, cell = self.encoder(src)
        input = trg[0, :]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[t] = output
            top1 = output.argmax(1)
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            input = trg[t] if teacher_force else top1

        return outputs

In [136]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_dim = len(eng_vocab)
output_dim = len(rus_vocab)

encoder = Encoder(input_dim, emb_dim=256, hidden_dim=1024)
decoder = Decoder(output_dim, emb_dim=256, hidden_dim=1024)

seq2seq_model = Seq2Seq(encoder, decoder, device).to(device)
seq2seq_model.load_state_dict(torch.load("best_model_2.pt", map_location=device))
seq2seq_model.eval()

print(seq2seq_model)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(16319, 256, padding_idx=0)
    (lstm): LSTM(256, 1024)
  )
  (decoder): Decoder(
    (embedding): Embedding(53282, 256, padding_idx=0)
    (lstm): LSTM(256, 1024)
    (fc): Linear(in_features=1024, out_features=53282, bias=True)
  )
)


In [137]:
def greedy_decode(sentence):
    return beam_search(seq2seq_model, sentence, eng_vocab, rus_vocab, device, beam_width=1, with_attention=False)

def beam_decode(sentence):
    return beam_search(seq2seq_model, sentence, eng_vocab, rus_vocab, device, beam_width=5, with_attention=False)

In [157]:
print("Evaluating Seq2Seq (greedy)")
bleu1, bleu2, bleu4 = evaluate_bleu(test_data, greedy_decode)

print("Seq2Seq Greedy BLEU-1:", round(bleu1, 4))
print("Seq2Seq Greedy BLEU-2:", round(bleu2, 4))
print("Seq2Seq Greedy BLEU-4:", round(bleu4, 4))

Evaluating Seq2Seq (greedy)
Seq2Seq Greedy BLEU-1: 0.6367
Seq2Seq Greedy BLEU-2: 0.5112
Seq2Seq Greedy BLEU-4: 0.2481


In [158]:
print("Evaluating Seq2Seq (beam search)")
bleu1_b, bleu2_b, bleu4_b = evaluate_bleu(test_data, beam_decode)

print("Seq2Seq Beam BLEU-1:", round(bleu1_b, 4))
print("Seq2Seq Beam BLEU-2:", round(bleu2_b, 4))
print("Seq2Seq Beam BLEU-4:", round(bleu4_b, 4))

Evaluating Seq2Seq (beam search)
Seq2Seq Beam BLEU-1: 0.6804
Seq2Seq Beam BLEU-2: 0.5617
Seq2Seq Beam BLEU-4: 0.3167


### Seq2Seq model with attention

In [140]:
import random

class AttentionEncoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, n_layers, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, n_layers, dropout=dropout if n_layers > 1 else 0)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.lstm(embedded)
        return outputs, hidden, cell


class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs, mask=None):
        src_len = encoder_outputs.shape[0]
        hidden = hidden[-1].unsqueeze(1).repeat(1, src_len, 1)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)
        if mask is not None:
            attention = attention.masked_fill(mask == 0, -1e10)
        return F.softmax(attention, dim=1)


class AttentionDecoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, attention, dropout=0.5):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim + hidden_dim, hidden_dim)
        self.fc_out = nn.Linear(emb_dim + hidden_dim + hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell, encoder_outputs, mask=None):
        input = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))
        a = self.attention(hidden, encoder_outputs, mask).unsqueeze(1)
        context = torch.bmm(a, encoder_outputs.permute(1, 0, 2)).permute(1, 0, 2)
        output, (hidden, cell) = self.rnn(torch.cat((embedded, context), dim=2), (hidden, cell))
        prediction = self.fc_out(torch.cat((output.squeeze(0), context.squeeze(0), embedded.squeeze(0)), dim=1))
        return prediction, hidden, cell


class Seq2SeqAttention(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5, mask=None):
        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        encoder_outputs, hidden, cell = self.encoder(src)
        input = trg[0]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell, encoder_outputs, mask)
            outputs[t] = output
            top1 = output.argmax(1)
            input = trg[t] if random.random() < teacher_forcing_ratio else top1

        return outputs

In [141]:
att_encoder = AttentionEncoder(input_dim, emb_dim=256, hidden_dim=512, n_layers=1)
attention = Attention(hidden_dim=512)
att_decoder = AttentionDecoder(output_dim, emb_dim=256, hidden_dim=512, attention=attention)

att_model = Seq2SeqAttention(att_encoder, att_decoder, device).to(device)
att_model.load_state_dict(torch.load("best_att_model_final.pt", map_location=device))
att_model.eval()

print(att_model)

Seq2SeqAttention(
  (encoder): AttentionEncoder(
    (embedding): Embedding(16319, 256)
    (lstm): LSTM(256, 512)
  )
  (decoder): AttentionDecoder(
    (attention): Attention(
      (attn): Linear(in_features=1024, out_features=512, bias=True)
      (v): Linear(in_features=512, out_features=1, bias=False)
    )
    (embedding): Embedding(53282, 256)
    (rnn): LSTM(768, 512)
    (fc_out): Linear(in_features=1280, out_features=53282, bias=True)
    (dropout): Dropout(p=0.5, inplace=False)
  )
)


In [151]:
def att_beam_decode(sentence):
    return beam_search(att_model, sentence, eng_vocab, rus_vocab, device, beam_width=3, with_attention=True)

In [156]:
print("Evaluating Seq2Seq with Attention")
bleu1_att, bleu2_att, bleu4_att = evaluate_bleu(test_data, att_beam_decode)

print("Attention Beam BLEU-1:", round(bleu1_att, 4))
print("Attention Beam BLEU-2:", round(bleu2_att, 4))
print("Attention Beam BLEU-4:", round(bleu4_att, 4))

Evaluating Seq2Seq with Attention
Attention Beam BLEU-1: 0.5691
Attention Beam BLEU-2: 0.4307
Attention Beam BLEU-4: 0.2636


### Summary

In [155]:
print("=" * 60)
print(f"{'Model':<35} {'BLEU-1':>6} {'BLEU-2':>6} {'BLEU-4':>6}")
print("=" * 60)
print(f"{'Seq2Seq (greedy)':<35} {bleu1:>6.4f} {bleu2:>6.4f} {bleu4:>6.4f}")
print(f"{'Seq2Seq (beam=5)':<35} {bleu1_b:>6.4f} {bleu2_b:>6.4f} {bleu4_b:>6.4f}")
print(f"{'Seq2Seq + Attention (beam=3)':<35} {bleu1_att:>6.4f} {bleu2_att:>6.4f} {bleu4_att:>6.4f}")
print("=" * 60)

Model                               BLEU-1 BLEU-2 BLEU-4
Seq2Seq (greedy)                    0.6367 0.5112 0.2481
Seq2Seq (beam=5)                    0.6804 0.5617 0.3167
Seq2Seq + Attention (beam=3)        0.5691 0.4307 0.2636


### Translation Examples from Test Data

In [154]:
for src, ref in test_data[:10]:
    print("EN:      ", "|", src)
    print("Greedy:  ", "|", " ".join(greedy_decode(src)))
    print("Beam:    ", "|", " ".join(beam_decode(src)))
    print("Att+Beam:", "|", " ".join(att_beam_decode(src)))
    print()

EN:       | the old castle stands on the hill
Greedy:   | старый замок замок замок
Beam:     | старый замок замок замок
Att+Beam: | замок замок на холме

EN:       | tom was afraid of mary
Greedy:   | том боялся мэри
Beam:     | том боялся мэри
Att+Beam: | мэри том

EN:       | it was a blow to us
Greedy:   | это было было ошибкой
Beam:     | это был нам нам
Att+Beam: | для нас ошибкой

EN:       | if possible id like to go home now
Greedy:   | если бы я хотел сейчас пойти сейчас домой
Beam:     | если бы я хотел сейчас пойти сейчас домой
Att+Beam: | если бы я хотел бы пойти домой

EN:       | i wasnt worried about tom
Greedy:   | я не беспокоился за тома
Beam:     | я не беспокоился за тома
Att+Beam: | я волновался за тома

EN:       | tom will be a little late
Greedy:   | том опоздает немного опоздает
Beam:     | том немного опоздает
Att+Beam: | будет немного

EN:       | i was always good at math
Greedy:   | я всегда был в полном настроении
Beam:     | я всегда был во всём
Att+Beam: